# 1.1 Reading a 2D scalar field from a simulation

First we define the objects we will use - the mesh reader (to read the mesh from the FEM file), the pickle manager (to save the mesh data that we want to analyse), and the unfolder (to manage the mesh).

In [1]:
import numpy as np

from cyclops.sim_reader import MeshReader
from cyclops.object_reader import PickleManager
from cyclops.regressors import LModel, CSModel
from cyclops.fields import ScalarField

# Define the necessary objects
reader = MeshReader("data/monoblock_out.e")
pickle_manager = PickleManager()


<meshio mesh object>
  Number of points: 87200
  Number of cells:
    hexahedron27: 2016
    hexahedron27: 3360
    hexahedron27: 4704
  Point sets: right, top, left, , centre_x_bottom_y_back_z, centre_x_bottom_y_front_z, left_x_bottom_y_centre_z, right_x_bottom_y_centre_z
  Point data: disp_x, disp_y, disp_z, temperature
  Cell data: vonmises_stress


In [ ]:
cell_list = MeshReader.get_boundary_faces()
print(cell_list)




[]


In [ ]:
# Read the simulation data
sensor_region = "right"
pos_3D = reader.read_pos(sensor_region)
pos_2D = unfolder.compress_2D(pos_3D)
bounds = unfolder.find_bounds(pos_2D)
grid = unfolder.generate_grid(bounds, 30, 30)

temps = reader.read_scalar(sensor_region, "temperature").reshape(-1, 1)
temp_field = ScalarField(LModel, bounds)
temp_field.fit_model(pos_2D, temps)

# Save the simulation data
pickle_manager.save_file("results/temp_plane_field.pickle", temp_field)
pickle_manager.save_file("results/temp_plane_points.pickle", grid)

We know that the field is uniform in the horizontal direction so there is no point in analysing a 2D scalar field when we could be using a 1D scalar field, so instead we compress it further into a line.

We create a new `ScalarField` only this time it is 1D.

In [ ]:
# Now compress our nice 2D field even further into a 1D field
pos1 = (bounds[0][0], bounds[0][1])
pos2 = (bounds[0][0], bounds[1][1])
line_2D = unfolder.generate_line(pos1, pos2, 50)
line_temps = temp_field.predict_values(line_2D)
line_1D = unfolder.compress_1D(line_2D)

bounds_1D = np.array([line_1D[0], line_1D[-1]])
new_line_field = ScalarField(CSModel, bounds_1D)
new_line_field.fit_model(line_1D, line_temps)

# Save the new 1D line field
pickle_manager.save_file("results/temp_line_field.pickle", new_line_field)
pickle_manager.save_file("results/temp_line_points.pickle", line_1D)